### **Sieć neuronowa MTL na reprezentacji fingerprint - zestaw 3 - Kardiotoksyczność**

Wykorzystana reprezentacja: **ECFP4**

Lista endpointów:


1. hERG (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. VDss (Lombardo)
5. AMES Mutagenicity

Wyniki dla STL:

In [1]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytdc 1.1.15 requires cellxgene-census==1.15.0, which is not installed.
pytdc 1.1.15 requires dataclasses<1.0,>=0.6, which is not installed.
pytdc 1.1.15 requires evaluate==0.4.2, which is not installed.
pytdc 1.1.15 requires gget<1.0.0,>=0.28.4, which is not installed.
pytdc 1.1.15 requires tiledbsoma<2.0.0,>=1.7.2, which is not installed.
pytdc 1.1.15 requires accelerate==0.33.0, but you have accelerate 1.13.0 which is incompatible.
pytdc 1.1.15 requires datasets<2.20.0, but you have datasets 4.0.0 which is incompatible.
pytdc 1.1.15 requires huggingface-hub<1.0,>=0.20.3, but you have huggingface-hub 1.16.1 which is incompatible.
pytdc 1.1.15 requires numpy<2.0.0,>=1.26.4, but you have numpy 2.4.6 which is incompatible.
pytdc 1.1.15 requires pandas<3.0.0,>=2.1.4, but you have pandas 3.0.3 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data_folder = "/content/drive/MyDrive/mldd_data" #folder z zapisanymi zestawami danych aby umożliwić porównanie danych na dokładnie tych samych zestawach dla każdego modelu

# data_folder = "/content/drive/MyDrive/MLDD - ADMET/data_splits"

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, accuracy_score, f1_score

In [5]:
class Featurizer:
    def __init__(self, y_column, smiles_col='Drug', **kwargs):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.__dict__.update(kwargs)
    def __call__(self, df):
        raise NotImplementedError()

class ECFPFeaturizer(Featurizer): #dodane zabezpieczenia na Nan etc
    def __init__(self, y_column='Y', radius=2, length=1024, **kwargs):
        self.radius = radius
        self.length = length
        super().__init__(y_column, **kwargs)

    def __call__(self, df):
        fingerprints = []
        labels = []
        for i, row in df.iterrows():
            smiles = row[self.smiles_col]
            mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None

            if mol:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, nBits=self.length)
                fingerprints.append(np.array(fp))
            else:
                # Zamiast pomijać wiersz, wstawiamy wektor zer (zachowuje wyrównanie)
                fingerprints.append(np.zeros(self.length))

            # Pobieramy etykietę jeśli istnieje (dla dummy X wstawiamy NaN)
            labels.append(row[self.y_column] if self.y_column in df.columns else np.nan)

        return np.array(fingerprints), np.array(labels).reshape(-1, 1)

In [6]:

# DEFINICJA SIECI NEURONOWEJ

class AdmetEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.res_layer = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = self.main(x)
        return torch.relu(x + self.res_layer(x))


class MTL_NN_Hybrid(nn.Module):
    def __init__(self, input_dim, reg_tasks, class_tasks, hidden_dim=512):
        """
        reg_tasks: lista nazw zadań regresyjnych (np. ['Lipophilicity', 'Solubility'])
        class_tasks: lista nazw zadań klasyfikacyjnych (np. ['BBB', 'Ames'])
        """
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.reg_tasks = reg_tasks
        self.class_tasks = class_tasks

        # Słowniki głowic dla każdego typu zadania
        #Pozwala zapisać wiele warstw Linear i nazwać je tak, jak nazywają się Twoje zadania.
        self.reg_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in reg_tasks
        })
        self.class_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in class_tasks
        })

    def forward(self, x):
        shared_features = self.encoder(x) #wspólny enkoder
        results = {}

        # Procesowanie zadań regresyjnych
        for task in self.reg_tasks:
            results[task] = self.reg_heads[task](shared_features)

        # Procesowanie zadań klasyfikacyjnych
        for task in self.class_tasks:
            # results[task] = torch.sigmoid(self.class_heads[task](shared_features))
            results[task] = self.class_heads[task](shared_features)

        #print(results)
        return results #Zwracając słownik:{'Caco2_Wang': 1.2, 'Lipophilicity_AZ': 0.5} masz absolutną pewność, który wynik dotyczy którego badania.

In [7]:

# --- STL REGRESOR ---
class STL_NN_Regressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1) # Wynik liniowy (dowolna liczba)

    def forward(self, x):
        return self.head(self.encoder(x))

# --- STL KLASYFIKATOR ---
class STL_NN_Classifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Sigmoid zamienia wynik na prawdopodobieństwo (0-1)
        return torch.sigmoid(self.head(self.encoder(x)))

  #=============================

In [8]:
def train_MTL_hybrid_wl_sum(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- 1. Normalizacja (osobny skaler dla każdego zadania regresyjnego) ---
    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        # Wyciągamy dane i maskujemy NaN, aby skaler zadziałał
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        # Testowe przesyłamy w oryginale (skalujemy tylko przy ewaluacji dla wygody)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        # Klasyfikacja nie wymaga skalowania
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    # --- 2. Model i Optymalizator ---
    model = MTL_NN_Hybrid(input_dim=X_train_np.shape[1], reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none') # 'none' pozwala nam ręcznie obsłużyć NaN
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task) ---")
    model.train()
    for epoch in range(500):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        # Sumujemy straty ze wszystkich zadań, ignorując NaN
        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    # --- 3. Ewaluacja ---
    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            # Odwracamy skalowanie
            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()


            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask]) # <-- Dodaj tę linię
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,  # <-- I tę linię
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [ ]:
def train_MTL_hybrid_uniform_average(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = MTL_NN_Hybrid(input_dim=X_train_np.shape[1], reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none')
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task - Uniform Average) ---")
    model.train()
    for epoch in range(500):
        optimizer.zero_grad()
        outputs = model(X_train_t)

        task_losses = []

        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        if task_losses:
            total_loss = torch.stack(task_losses).mean() # Uniform average
        else:
            total_loss = torch.tensor(0.0, device=device, requires_grad=True)

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [ ]:
def train_MTL_hybrid_uncertainty_weighting(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    # 1. Przygotowanie danych i skalerów (z zachowaniem oryginalnych masek NaN)
    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    # Inicjalizacja modelu
    model = MTL_NN_Hybrid(input_dim=X_train_np.shape[1], reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)

    # 2. Poprawna inicjalizacja parametrów Uncertainty
    log_vars = nn.ParameterDict()
    for task in reg_tasks:
        log_vars[task] = nn.Parameter(torch.zeros(1, device=device))  # Regresja (MSE startuje z e^0 = 1)
    for task in class_tasks:
        log_vars[task] = nn.Parameter(torch.ones(1, device=device) * 0.5)  # Klasyfikacja (BCE wymaga mniejszej wagi na start)

    # 3. Wolniejszy LR + Weight Decay (Klucz do generalizacji w MTL)
    optimizer = optim.Adam(list(model.parameters()) + list(log_vars.values()), lr=0.0003, weight_decay=1e-4)

    criterion_reg = nn.MSELoss(reduction='none')
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Zoptymalizowany Uncertainty Weighting MTL) ---")
    model.train()
    for epoch in range(500):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        # --- ZADANIA REGRESYJNE ---
        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                # Obliczamy czysty średni błąd tylko dla obecnych próbek laboratoryjnych
                base_loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                precision = torch.exp(-log_vars[task])
                # Oryginalne równanie Kendalla dla regresji
                total_loss += 0.5 * precision * base_loss + 0.5 * log_vars[task]

        # --- ZADANIA KLASYFIKACYJNE ---
        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                # Obliczamy czysty średni błąd tylko dla obecnych próbek laboratoryjnych
                base_loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                precision = torch.exp(-log_vars[task])
                # Poprawne równanie Kendalla dla klasyfikacji binarnej
                total_loss += precision * base_loss + 0.5 * log_vars[task]

        total_loss.backward()

        # 4. Ochrona enkodera przed sprzecznymi gradientami (Gradient Clipping)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        if epoch % 50 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    print("\n--- EWALUACJA (Na czystych danych testowych) ---\n")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [9]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [10]:
def print_metrics(metrics, task='classification', weight_loss_func_name=None):
    print(f"\n{'='*40}")
    if weight_loss_func_name: # Check if func_name is provided and not None
        print(f"  Loss Weighting: {weight_loss_func_name}")
        print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification', weight_loss_func_name=None, endpoint_group_name=None):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        if endpoint_group_name:
            f.write(f"Tasks       : {endpoint_group_name}\n")
        if weight_loss_func_name:
            f.write(f"Loss Weighting: {weight_loss_func_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [11]:
import numpy as np
import pandas as pd

def prepare_mtl_data_embeddings(df_list, task_names, dataset_names):
    """
    df_list: lista DataFrame ze splitów (np. [train_caco2, train_lipo])
    task_names: lista nazw zadań w modelu (np. ['Caco2_Wang', 'Lipophilicity_AZ'])
    dataset_names: lista nazw zbiorów odpowiadająca nazwom plików z embeddingami (np. ['Caco2_Wang', 'Lipophilicity_AstraZeneca'])
    """
    # 1. Zebranie wszystkich unikalnych struktur SMILES z przekazanych splitów
    all_drugs = set()
    for df in df_list:
        valid = df['Drug'].dropna().astype(str).unique()
        all_drugs.update(valid)

    # 2. Wczytanie embeddingów ze wszystkich powiązanych plików CSV dla danej konfiguracji
    # Budujemy jeden zbiorczy słownik mapujący Drug (SMILES) -> wektor embeddingu
    drug_to_emb = {}
    for d_name in dataset_names:
        file_path = f"{data_folder}/{d_name}_MoLFormer_embeddings.csv"
        try:
            emb_df = pd.read_csv(file_path)
            feature_cols = [c for c in emb_df.columns if c.startswith('emb_')]
            for _, row in emb_df.iterrows():
                drug_to_emb[str(row['Drug'])] = row[feature_cols].values.astype(np.float32)
        except Exception as e:
            print(f"Błąd podczas ładowania embeddingów dla {d_name}: {e}")

    # 3. Filtrowanie master_list tylko do cząsteczek, które posiadają embeddingi
    safe_master_list = [drug for drug in sorted(list(all_drugs)) if drug in drug_to_emb]
    drug_to_idx = {drug: i for i, drug in enumerate(safe_master_list)}
    n_samples = len(safe_master_list)

    # 4. Tworzenie macierzy X_features z embeddingów
    emb_dim = len(next(iter(drug_to_emb.values()))) if drug_to_emb else 768
    X_features = np.zeros((n_samples, emb_dim), dtype=np.float32)
    for i, drug in enumerate(safe_master_list):
        X_features[i] = drug_to_emb[drug]

    # 5. Mapowanie etykiet y (dokładnie tak samo jak w wersji z deskryptorami)
    y_dict = {}
    for df, task in zip(df_list, task_names):
        y_vec = np.full((n_samples, 1), np.nan, dtype=np.float32)
        mapping = dict(zip(df['Drug'].astype(str), df['Y']))

        for drug, val in mapping.items():
            if drug in drug_to_idx and not pd.isna(val):
                y_vec[drug_to_idx[drug]] = val
        y_dict[task] = y_vec

    return X_features, y_dict

In [12]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import copy

def train_MTL_alternating_batches(X_train_dict, y_train_dict, X_test_dict, y_test_dict, reg_tasks, class_tasks, batch_size=64, epochs=300):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Przygotowanie czystych DataLoaderów dla każdego zadania osobno
    dataloaders = {}
    scalers = {}

    # Przetwarzanie regresji
    for task in reg_tasks:
        scalers[task] = StandardScaler()
        X_task = X_train_dict[task]
        y_task = y_train_dict[task].reshape(-1, 1)

        # Filtrujemy ewentualne braki na starcie
        mask = ~np.isnan(y_task).flatten()
        X_clean = X_task[mask]
        y_clean = scalers[task].fit_transform(y_task[mask])

        dataset = TensorDataset(torch.FloatTensor(X_clean), torch.FloatTensor(y_clean))
        dataloaders[task] = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    # Przetwarzanie klasyfikacji
    for task in class_tasks:
        X_task = X_train_dict[task]
        y_task = y_train_dict[task].reshape(-1, 1)

        mask = ~np.isnan(y_task).flatten()
        X_clean = X_task[mask]
        y_clean = y_task[mask]

        dataset = TensorDataset(torch.FloatTensor(X_clean), torch.FloatTensor(y_clean))
        dataloaders[task] = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    # Inicjalizacja hybrydy (zakładamy, że znasz wymiar wejściowy np. 768 z pierwszego zadania)
    first_task = (reg_tasks + class_tasks)[0]
    input_dim = X_train_dict[first_task].shape[1]
    model = MTL_NN_Hybrid(input_dim=input_dim, reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)

    # Inicjalizacja parametrów Uncertainty Weighting
    log_vars = nn.ParameterDict()
    for task in reg_tasks: log_vars[task] = nn.Parameter(torch.zeros(1, device=device))
    for task in class_tasks: log_vars[task] = nn.Parameter(torch.ones(1, device=device) * 0.5)

    # Optymalizator z wyższą regularyzacją (Weight Decay)
    optimizer = optim.Adam(list(model.parameters()) + list(log_vars.values()), lr=0.0003, weight_decay=1e-4)
    criterion_reg = nn.MSELoss()
    criterion_cls = nn.BCEWithLogitsLoss()

    print("\n--- START NOWEGO PROCESU UCZENIA (Alternating Mini-Batches MTL) ---")

    for epoch in range(epochs):
        model.train()

        # Tworzymy iteratory dla każdego zadania
        task_iters = {task: iter(dl) for task, dl in dataloaders.items()}

        # Wyznaczamy liczbę kroków w epoce na podstawie NAJWIĘKSZEGO zbioru,
        # aby mniejsze zadania nie skończyły się za szybko (zapętlamy je)
        max_steps = max(len(dl) for dl in dataloaders.values())

        for step in range(max_steps):
            optimizer.zero_grad()
            total_loss = 0

            # --- SEKCJA REGRESJI ---
            for task in reg_tasks:
                try:
                    bx, by = next(task_iters[task])
                except StopIteration:
                    # Jeśli skończyły się dane w mniejszym zbiorze, restartujemy iterator (Oversampling)
                    task_iters[task] = iter(dataloaders[task])
                    bx, by = next(task_iters[task])

                bx, by = bx.to(device), by.to(device)

                # Przepychamy przez wspólny enkoder w tym konkretnym kroku
                features = model.encoder(bx)
                pred = model.reg_heads[task](features)

                base_loss = criterion_reg(pred, by)
                precision = torch.exp(-log_vars[task])
                total_loss += 0.5 * precision * base_loss + 0.5 * log_vars[task]

            # --- SEKCJA KLASYFIKACJI ---
            for task in class_tasks:
                try:
                    bx, by = next(task_iters[task])
                except StopIteration:
                    task_iters[task] = iter(dataloaders[task])
                    bx, by = next(task_iters[task])

                bx, by = bx.to(device), by.to(device)

                features = model.encoder(bx)
                pred = model.class_heads[task](features)

                base_loss = criterion_cls(pred, by)
                precision = torch.exp(-log_vars[task])
                total_loss += precision * base_loss + 0.5 * log_vars[task]

            # Wspólny krok wsteczny - akumulacja zbalansowanych gradientów
            total_loss.backward()

            # Ochrona przed wzajemnym niszczeniem wag (Gradient Clipping)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Przykładowy Total Loss z kroku: {total_loss.item():.4f}")

    # --- EWALUACJA ---
    print("\n--- EWALUACJA NA ZBIORACH TESTOWYCH ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        for task in reg_tasks:
            X_test_t = torch.FloatTensor(X_test_dict[task]).to(device)
            y_true = y_test_dict[task].flatten()

            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            features = model.encoder(X_test_t)
            p_scaled = model.reg_heads[task](features).cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mean_absolute_error(y_true[mask], p_unscaled[mask]),
               "r2": r2_score(y_true[mask], p_unscaled[mask])
             }

        for task in class_tasks:
            X_test_t = torch.FloatTensor(X_test_dict[task]).to(device)
            y_true = y_test_dict[task].flatten()

            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            features = model.encoder(X_test_t)
            p_logits = model.class_heads[task](features)
            p_probs = torch.sigmoid(p_logits).cpu().numpy().flatten()
            p_labels = (p_probs > 0.5).astype(int)

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels[mask]),
                "f1":       f1_score(y_true[mask], p_labels[mask]),
                "auroc":    roc_auc_score(y_true[mask], p_probs[mask]),
            }

    return all_metrics

In [13]:
def prepare_clean_dicts_for_alternating(df_list, task_names, dataset_names):
    """
    Przygotowuje oddzielne, czyste macierze X i y dla każdego zadania,
    eliminując potrzebę stosowania unii z wartościami NaN.
    """
    X_dict = {}
    y_dict = {}

    for df, task, d_name in zip(df_list, task_names, dataset_names):
        file_path = f"{data_folder}/{d_name}_MoLFormer_embeddings.csv"
        try:
            emb_df = pd.read_csv(file_path)
            feature_cols = [c for c in emb_df.columns if c.startswith('emb_')]

            # Mapujemy embeddingi bezpośrednio dla cząsteczek z danego splitu
            drug_to_emb = {str(row['Drug']): row[feature_cols].values.astype(np.float32) for _, row in emb_df.iterrows()}

            clean_features = []
            clean_labels = []

            for _, row in df.iterrows():
                drug = str(row['Drug'])
                if drug in drug_to_emb and not pd.isna(row['Y']):
                    clean_features.append(drug_to_emb[drug])
                    clean_labels.append(row['Y'])

            X_dict[task] = np.array(clean_features, dtype=np.float32)
            y_dict[task] = np.array(clean_labels, dtype=np.float32).reshape(-1, 1)

        except Exception as e:
            print(f"Błąd ładowania danych dla zadania {task} ({d_name}): {e}")

    return X_dict, y_dict

In [14]:
def train_STL_classifier_low_data(X_train_np, y_train_np, X_test_np, y_test_np, epochs=101):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Przygotowanie danych treningowych jako pojedynczy tensor
    X_train_t = torch.FloatTensor(X_train_np).to(device)
    y_train_t = torch.FloatTensor(y_train_np).to(device)

    # Inicjalizacja modelu STL (wykorzystuje ten sam AdmetEncoder co MTL)
    model_stl = STL_NN_Classifier(input_dim=X_train_np.shape[1]).to(device)
    optimizer = optim.Adam(model_stl.parameters(), lr=0.0005) # Zmiana LR i usunięcie weight_decay
    criterion = nn.BCELoss() # Nasz STL ma wbudowany Sigmoid w klasie STL_NN_Classifier

    model_stl.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        out = model_stl(X_train_t)
        loss = criterion(out, y_train_t)
        loss.backward()
        # Usunięto gradient clipping, aby działało jak w train_NN_classification
        optimizer.step()

    # Ewaluacja STL
    model_stl.eval()
    with torch.no_grad():
        X_test_t = torch.FloatTensor(X_test_np).to(device)
        p_probs = model_stl(X_test_t).cpu().numpy().flatten()
        p_labels = (p_probs > 0.5).astype(int)

        metrics = {
            "accuracy": accuracy_score(y_test_np, p_labels),
            "f1":       f1_score(y_test_np, p_labels),
            "auroc":    roc_auc_score(y_test_np, p_probs)
        }
    return metrics

In [35]:
# def train_STL_classifier_low_data(X_train_clean, y_train_clean, X_test_clean, y_test_clean, epochs=300, batch_size=64):
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#     # Przygotowanie DataLoaderu dla STL
#     dataset = TensorDataset(torch.FloatTensor(X_train_clean), torch.FloatTensor(y_train_clean))
#     dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
#     print(len(dataloader))

#     # Inicjalizacja modelu STL (wykorzystuje ten sam AdmetEncoder co MTL)
#     model_stl = STL_NN_Classifier(input_dim=X_train_clean.shape[1]).to(device)
#     optimizer = optim.Adam(model_stl.parameters(), lr=0.0003, weight_decay=1e-4)
#     criterion = nn.BCELoss() # Nasz STL ma wbudowany Sigmoid w klasie STL_NN_Classifier

#     print("\n  [STL] Start treningu klasyfikatora hERG na 10% danych...")
#     model_stl.train()
#     for epoch in range(epochs):
#         for bx, by in dataloader:
#             bx, by = bx.to(device), by.to(device)
#             optimizer.zero_grad()
#             out = model_stl(bx)
#             loss = criterion(out, by)
#             loss.backward()
#             torch.nn.utils.clip_grad_norm_(model_stl.parameters(), max_norm=1.0)
#             optimizer.step()

#     # Ewaluacja STL
#     model_stl.eval()
#     with torch.no_grad():
#         X_test_t = torch.FloatTensor(X_test_clean).to(device)
#         p_probs = model_stl(X_test_t).cpu().numpy().flatten()
#         p_labels = (p_probs > 0.5).astype(int)

#         metrics = {
#             "accuracy": accuracy_score(y_test_clean, p_labels),
#             "f1":       f1_score(y_test_clean, p_labels),
#             "auroc":    roc_auc_score(y_test_clean, p_probs)
#         }
#     return metrics

In [19]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler # Added import for StandardScaler

def train_STL_regressor_low_data(X_train_np, y_train_np, X_test_np, y_test_np):

    # C. Normalizacja etykiet
    print("[INFO] Normalizacja danych wyjściowych...")
    scaler = StandardScaler()
    y_train_scaled = scaler.fit_transform(y_train_np.reshape(-1, 1))
    y_test_scaled = scaler.transform(y_test_np.reshape(-1, 1))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.FloatTensor(X_train_np).to(device)
    y_train_t = torch.FloatTensor(y_train_scaled).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = STL_NN_Regressor(input_dim=X_train_np.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0005)

    print("\n--- START TRENINGU ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            mse_val = loss.item()
            print(f"  Epoka {epoch:3d} | Błąd MSE: {loss.item():.4f}")

# F. Ewaluacja
    print("\n--- EWALUACJA ---")
    model.eval()
    with torch.no_grad():
        preds_scaled = model(X_test_t).cpu().numpy()

    preds = scaler.inverse_transform(preds_scaled)

    mse = mean_squared_error(y_test_np, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_np, preds)
    r2 = r2_score(y_test_np, preds)

    metrics = {
        "test_metrics": {
            "rmse": rmse,
            "mae": mae,
            "r2": r2
        }
    }

    return metrics

In [15]:
def train_MTL_alternating_batches(X_train_dict, y_train_dict, X_test_dict, y_test_dict, reg_tasks, class_tasks, batch_size=32, epochs=300):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataloaders = {}
    scalers = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        X_task = X_train_dict[task]
        y_task = y_train_dict[task].reshape(-1, 1)

        mask = ~np.isnan(y_task).flatten()
        X_clean = X_task[mask]
        y_clean = scalers[task].fit_transform(y_task[mask])

        dataset = TensorDataset(torch.FloatTensor(X_clean), torch.FloatTensor(y_clean))
        # ZMIANA: drop_last=False, aby nie usunąć małych zbiorów danych
        dataloaders[task] = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)

    for task in class_tasks:
        X_task = X_train_dict[task]
        y_task = y_train_dict[task].reshape(-1, 1)

        mask = ~np.isnan(y_task).flatten()
        X_clean = X_task[mask]
        y_clean = y_task[mask]

        dataset = TensorDataset(torch.FloatTensor(X_clean), torch.FloatTensor(y_clean))
        # ZMIANA: drop_last=False, aby nie usunąć małych zbiorów danych
        dataloaders[task] = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)

    input_dim = X_train_dict[(reg_tasks + class_tasks)[0]].shape[1]
    model = MTL_NN_Hybrid(input_dim=input_dim, reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)

    log_vars = nn.ParameterDict()
    for task in reg_tasks: log_vars[task] = nn.Parameter(torch.zeros(1, device=device))
    for task in class_tasks: log_vars[task] = nn.Parameter(torch.ones(1, device=device) * 0.5)

    optimizer = optim.Adam(list(model.parameters()) + list(log_vars.values()), lr=0.0003, weight_decay=1e-4)
    criterion_reg = nn.MSELoss()
    criterion_cls = nn.BCEWithLogitsLoss()

    # Generator tworzący nieskończony strumień danych
    def infinite_batch_provider(dataloader):
        while True:
            for batch in dataloader:
                yield batch

    print("\n--- START UCZENIA (Alternating Mini-Batches MTL - Bezpieczny Strumień) ---")

    for epoch in range(epochs):
        model.train()
        task_iters = {task: infinite_batch_provider(dataloaders[task]) for task in dataloaders}
        max_steps = max(len(dl) for dl in dataloaders.values())

        for step in range(max_steps):
            optimizer.zero_grad()
            total_loss = 0

            # --- SEKCJA REGRESJI ---
            for task in reg_tasks:
                bx, by = next(task_iters[task])
                bx, by = bx.to(device), by.to(device)

                features = model.encoder(bx)
                pred = model.reg_heads[task](features)

                base_loss = criterion_reg(pred, by)
                precision = torch.exp(-log_vars[task])
                total_loss += 0.5 * precision * base_loss + 0.5 * log_vars[task]

            # --- SEKCJA KLASYFIKACJI ---
            for task in class_tasks:
                bx, by = next(task_iters[task])
                bx, by = bx.to(device), by.to(device)

                features = model.encoder(bx)
                pred = model.class_heads[task](features)

                base_loss = criterion_cls(pred, by)
                precision = torch.exp(-log_vars[task])
                total_loss += precision * base_loss + 0.5 * log_vars[task]

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Przykładowy Total Loss z kroku: {total_loss.item():.4f}")

    # --- EWALUACJA ---
    print("\n--- EWALUACJA NA ZBIORACH TESTOWYCH ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        for task in reg_tasks:
            X_test_t = torch.FloatTensor(X_test_dict[task]).to(device)
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            features = model.encoder(X_test_t)
            p_scaled = model.reg_heads[task](features).cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mean_absolute_error(y_true[mask], p_unscaled[mask]),
               "r2": r2_score(y_true[mask], p_unscaled[mask])
             }

        for task in class_tasks:
            X_test_t = torch.FloatTensor(X_test_dict[task]).to(device)
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            features = model.encoder(X_test_t)
            p_logits = model.class_heads[task](features)
            p_probs = torch.sigmoid(p_logits).cpu().numpy().flatten()
            p_labels = (p_probs > 0.5).astype(int)

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels[mask]),
                "f1":       f1_score(y_true[mask], p_labels[mask]),
                "auroc":    roc_auc_score(y_true[mask], p_probs[mask]),
            }

    return all_metrics

In [16]:
import numpy as np
import pandas as pd
from sklearn.utils import resample

# Ścieżka do zapisu wyników z symulacji
filepath_sim = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_embeddings_sym.txt"

reg_tasks = ['Lipophilicity_AZ']
class_tasks = ['hERG']

# 1. Ładowanie oryginalnych splitów (poza pętlą, bo są stałe)
train_lipo_orig, test_lipo_orig = load_split_pickle('Lipophilicity_AstraZeneca')
train_herg_orig, test_herg_orig = load_split_pickle('hERG')

hERG_percentages = [0.10, 0.50, 1.00] # Dodane 50% i 100%

for percentage in hERG_percentages:
    print("\n" + "="*60)
    print(f"  START EMULACJI GŁODU DANYCH ({int(percentage*100)}% hERG vs 100% Lipofilowość)")
    print("="*60)

    # Sztuczne obcięcie zbioru TRENINGOWEGO hERG do `percentage`
    n_samples_percentage = int(len(train_herg_orig) * percentage)
    train_herg_simulated = resample(train_herg_orig, replace=False, n_samples=n_samples_percentage, random_state=42)

    print(f" -> Oryginalny rozmiar treningowy hERG: {len(train_herg_orig)} cząsteczek")
    print(f" -> Symulowany ({int(percentage*100)}%) rozmiar treningowy hERG: {len(train_herg_simulated)} cząsteczek")
    print(f" -> Rozmiar treningowy Lipofilowości (100%): {len(train_lipo_orig)} cząsteczek")

    # 2. Przygotowanie czystych słowników pod model MTL (używamy zredukowanego zbioru hERG)
    X_train_dict, y_train_dict = prepare_clean_dicts_for_alternating(
        [train_lipo_orig, train_herg_simulated], reg_tasks + class_tasks, ['Lipophilicity_AstraZeneca', 'hERG']
    )
    # Testowe zbiory danych pozostają takie same dla wszystkich eksperymentów
    X_test_dict,  y_test_dict  = prepare_clean_dicts_for_alternating(
        [test_lipo_orig, test_herg_orig],  reg_tasks + class_tasks, ['Lipophilicity_AstraZeneca', 'hERG']
    )

    # =====================================================================
    # KROK A: Uruchomienie modelu STL (Single-Task) na małych danych
    # =====================================================================
    # Pobieramy czyste, odseparowane dane hERG dla modelu STL
    X_tr_herg_stl = X_train_dict['hERG']
    y_tr_herg_stl = y_train_dict['hERG']
    X_te_herg_stl = X_test_dict['hERG']
    y_te_herg_stl = y_test_dict['hERG']

    stl_herg_metrics = train_STL_classifier_low_data(X_tr_herg_stl, y_tr_herg_stl, X_te_herg_stl, y_te_herg_stl)


    # =====================================================================
    # KROK B: Uruchomienie Twojego modelu MTL (Alternating Batches)
    # =====================================================================
    # Model MTL dostaje dokładnie ten sam mały wycinek hERG, ale dodatkowo widzi całą Lipofilowość
    mtl_results = train_MTL_alternating_batches(X_train_dict, y_train_dict, X_test_dict, y_test_dict, reg_tasks, class_tasks, batch_size=64, epochs=300)


    # =====================================================================
    # KROK C: BEZPOŚREDNIE PORÓWNANIE I RAPORT
    # =====================================================================
    mtl_herg_metrics = mtl_results['class_tasks']['hERG']

    print("\n" + "="*50)
    print(f"   WYNIKI KOŃCOWE SYMULACJI (DLA ZADANIA hERG - {int(percentage*100)}%)")
    print("="*50)
    print(f" METRYKA    |  MODEL STL ({int(percentage*100)}% danych)  |  MODEL MTL ({int(percentage*100)}% danych + Lipo)")
    print("-"*50)
    print(f" AUROC      |          {stl_herg_metrics['auroc']:.4f}          |          {mtl_herg_metrics['auroc']:.4f}")
    print(f" F1-Score   |          {stl_herg_metrics['f1']:.4f}          |          {mtl_herg_metrics['f1']:.4f}")
    print(f" Accuracy   |          {stl_herg_metrics['accuracy']:.4f}          |          {mtl_herg_metrics['accuracy']:.4f}")
    print("="*50)

    # Zapis wyników symulacji do Twojego pliku tekstowego jako dowód naukowy
    with open(filepath_sim, 'a') as f:
        f.write(f"\n\n{'='*50}\nEKSPERYMENT KONTROLNY: SYMULACJA GŁODU DANYCH ({int(percentage*100)}% hERG)\n{'='*50}\n")
        f.write(f"  [STL hERG {int(percentage*100)}%] AUROC: {stl_herg_metrics['auroc']:.4f} | F1: {stl_herg_metrics['f1']:.4f} | ACC: {stl_herg_metrics['accuracy']:.4f}\n")
        f.write(f"  [MTL hERG {int(percentage*100)}%] AUROC: {mtl_herg_metrics['auroc']:.4f} | F1: {mtl_herg_metrics['f1']:.4f} | ACC: {mtl_herg_metrics['accuracy']:.4f}\n")
        f.write(f"{'='*50}\n")

print(f"\nWszystkie wyniki symulacji zostały pomyślnie dopisane do pliku:\n{filepath_sim}")


  START EMULACJI GŁODU DANYCH (10% hERG vs 100% Lipofilowość)
 -> Oryginalny rozmiar treningowy hERG: 524 cząsteczek
 -> Symulowany (10%) rozmiar treningowy hERG: 52 cząsteczek
 -> Rozmiar treningowy Lipofilowości (100%): 3360 cząsteczek

--- START UCZENIA (Alternating Mini-Batches MTL - Bezpieczny Strumień) ---
  Epoka   0 | Przykładowy Total Loss z kroku: 0.7269
  Epoka  20 | Przykładowy Total Loss z kroku: -0.0676
  Epoka  40 | Przykładowy Total Loss z kroku: -0.3927
  Epoka  60 | Przykładowy Total Loss z kroku: -0.7124
  Epoka  80 | Przykładowy Total Loss z kroku: -1.0099
  Epoka 100 | Przykładowy Total Loss z kroku: -1.3557
  Epoka 120 | Przykładowy Total Loss z kroku: -1.6530
  Epoka 140 | Przykładowy Total Loss z kroku: -1.9524
  Epoka 160 | Przykładowy Total Loss z kroku: -2.1850
  Epoka 180 | Przykładowy Total Loss z kroku: -2.4694
  Epoka 200 | Przykładowy Total Loss z kroku: -2.7519
  Epoka 220 | Przykładowy Total Loss z kroku: -3.1015
  Epoka 240 | Przykładowy Total Loss z

In [ ]:
import numpy as np
import pandas as pd
from sklearn.utils import resample

filepath_sim = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_embeddings_sym.txt"

reg_tasks = ['Caco2_Wang', 'Lipophilicity_AZ']
class_tasks = []

# 1. Ładowanie oryginalnych splitów
train_caco2_orig, test_caco2_orig = load_split_pickle('Caco2_Wang')
train_lipo_orig, test_lipo_orig = load_split_pickle('Lipophilicity_AstraZeneca')

low_data_percentages = [0.10, 0.50, 1.00]

for percentage in low_data_percentages:
    print("\n" + "="*60)
    print(f"  START EMULACJI GŁODU DANYCH ( {int(percentage*100)}% Caco2_Wang vs 100% Lipofilowość)")
    print("="*60)

    # Sztuczne obcięcie zbioru TRENINGOWEGO Caco2_Wang do `percentage`
    n_samples_percentage = int(len(train_caco2_orig) * percentage)
    train_caco2_simulated = resample(train_caco2_orig, replace=False, n_samples=n_samples_percentage, random_state=42)

    print(f" -> Oryginalny rozmiar treningowy Caco2_Wang: {len(train_caco2_orig)} cząsteczek")
    print(f" -> Symulowany ({int(percentage*100)}%) rozmiar treningowy Caco2_Wang: {len(train_caco2_simulated)} cząsteczek")
    print(f" -> Rozmiar treningowy Lipofilowości (100%): {len(train_lipo_orig)} cząsteczek")

    # 2. Przygotowanie czystych słowników pod model MTL
    X_train_dict, y_train_dict = prepare_clean_dicts_for_alternating(
        [train_caco2_simulated, train_lipo_orig],
        reg_tasks + class_tasks,
        ['Caco2_Wang', 'Lipophilicity_AstraZeneca']
    )
    # Testowe zbiory danych pozostają takie same dla wszystkich eksperymentów
    X_test_dict,  y_test_dict  = prepare_clean_dicts_for_alternating(
        [test_caco2_orig, test_lipo_orig],
        reg_tasks + class_tasks,
        ['Caco2_Wang', 'Lipophilicity_AstraZeneca']
    )

    # =====================================================================
    # KROK A: Uruchomienie modelu STL (Single-Task) na małych danych
    # =====================================================================
    X_tr_caco2_stl = X_train_dict['Caco2_Wang']
    y_tr_caco2_stl = y_train_dict['Caco2_Wang']
    X_te_caco2_stl = X_test_dict['Caco2_Wang']
    y_te_caco2_stl = y_test_dict['Caco2_Wang']

    stl_caco2_metrics = train_STL_regressor_low_data(X_tr_caco2_stl, y_tr_caco2_stl, X_te_caco2_stl, y_te_caco2_stl, epochs=300, batch_size=64)


    # =====================================================================
    # KROK B: Uruchomienie Twojego modelu MTL (Alternating Batches)
    # =====================================================================
    mtl_results = train_MTL_alternating_batches(X_train_dict, y_train_dict, X_test_dict, y_test_dict, reg_tasks, class_tasks, batch_size=64, epochs=300)


    # =====================================================================
    # KROK C: BEZPOŚREDNIE PORÓWNANIE I RAPORT
    # =====================================================================
    mtl_caco2_metrics = mtl_results['reg_tasks']['Caco2_Wang']

    print("\n" + "="*50)
    print(f"   WYNIKI KOŃCOWE SYMULACJI (DLA ZADANIA Caco2_Wang - {int(percentage*100)}%)")
    print("="*50)
    print(f" METRYKA    |  MODEL STL ({int(percentage*100)}% danych)  |  MODEL MTL ({int(percentage*100)}% danych + Lipo)")
    print("-"*50)
    print(f" RMSE       |          {stl_caco2_metrics['rmse']:.4f}          |          {mtl_caco2_metrics['rmse']:.4f}")
    print(f" MAE        |          {stl_caco2_metrics['mae']:.4f}          |          {mtl_caco2_metrics['mae']:.4f}")
    print(f" R2         |          {stl_caco2_metrics['r2']:.4f}          |          {mtl_caco2_metrics['r2']:.4f}")
    print("="*50)

    with open(filepath_sim, 'a') as f:
        f.write(f"\n\n{'='*50}\nEKSPERYMENT KONTROLNY: SYMULACJA GŁODU DANYCH ({int(percentage*100)}% Caco2_Wang)\n{'='*50}\n")
        f.write(f"  [STL Caco2_Wang {int(percentage*100)}%] RMSE: {stl_caco2_metrics['rmse']:.4f} | MAE: {stl_caco2_metrics['mae']:.4f} | R2: {stl_caco2_metrics['r2']:.4f}\n")
        f.write(f"  [MTL Caco2_Wang {int(percentage*100)}%] RMSE: {mtl_caco2_metrics['rmse']:.4f} | MAE: {mtl_caco2_metrics['mae']:.4f} | R2: {mtl_caco2_metrics['r2']:.4f}\n")
        f.write(f"{'='*50}\n")

print(f"\nWszystkie wyniki symulacji dla Caco2_Wang zostały pomyślnie dopisane do pliku:\n{filepath_sim}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.utils import resample

filepath_sim = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_embeddings_sym.txt"

reg_tasks = ['VDss_Lombardo', 'Lipophilicity_AZ']
class_tasks = []

# 1. Ładowanie oryginalnych splitów
train_vdss_orig, test_vdss_orig = load_split_pickle('VDss_Lombardo')
train_lipo_orig, test_lipo_orig = load_split_pickle('Lipophilicity_AstraZeneca')

low_data_percentages = [0.10, 0.50, 1.00]

for percentage in low_data_percentages:
    print("\n" + "="*60)
    print(f"  START EMULACJI GŁODU DANYCH ( {int(percentage*100)}% VDss_Lombardo vs 100% Lipofilowość)")
    print("="*60)

    # Sztuczne obcięcie zbioru TRENINGOWEGO VDss_Lombardo do `percentage`
    n_samples_percentage = int(len(train_vdss_orig) * percentage)
    train_vdss_simulated = resample(train_vdss_orig, replace=False, n_samples=n_samples_percentage, random_state=42)

    print(f" -> Oryginalny rozmiar treningowy VDss_Lombardo: {len(train_vdss_orig)} cząsteczek")
    print(f" -> Symulowany ({int(percentage*100)}%) rozmiar treningowy VDss_Lombardo: {len(train_vdss_simulated)} cząsteczek")
    print(f" -> Rozmiar treningowy Lipofilowości (100%): {len(train_lipo_orig)} cząsteczek")

    # 2. Przygotowanie czystych słowników pod model MTL
    X_train_dict, y_train_dict = prepare_clean_dicts_for_alternating(
        [train_vdss_simulated, train_lipo_orig],
        reg_tasks + class_tasks,
        ['VDss_Lombardo', 'Lipophilicity_AstraZeneca']
    )
    # Testowe zbiory danych pozostają takie same dla wszystkich eksperymentów
    X_test_dict,  y_test_dict  = prepare_clean_dicts_for_alternating(
        [test_vdss_orig, test_lipo_orig],
        reg_tasks + class_tasks,
        ['VDss_Lombardo', 'Lipophilicity_AstraZeneca']
    )

    # =====================================================================
    # KROK A: Uruchomienie modelu STL (Single-Task) na małych danych
    # =====================================================================
    X_tr_vdss_stl = X_train_dict['VDss_Lombardo']
    y_tr_vdss_stl = y_train_dict['VDss_Lombardo']
    X_te_vdss_stl = X_test_dict['VDss_Lombardo']
    y_te_vdss_stl = y_test_dict['VDss_Lombardo']

    stl_vdss_metrics = train_STL_regressor_low_data(X_tr_vdss_stl, y_tr_vdss_stl, X_te_vdss_stl, y_te_vdss_stl, epochs=300, batch_size=64)


    # =====================================================================
    # KROK B: Uruchomienie Twojego modelu MTL (Alternating Batches)
    # =====================================================================
    mtl_results = train_MTL_alternating_batches(X_train_dict, y_train_dict, X_test_dict, y_test_dict, reg_tasks, class_tasks, batch_size=64, epochs=300)


    # =====================================================================
    # KROK C: BEZPOŚREDNIE PORÓWNANIE I RAPORT
    # =====================================================================
    mtl_vdss_metrics = mtl_results['reg_tasks']['VDss_Lombardo']

    print("\n" + "="*50)
    print(f"   WYNIKI KOŃCOWE SYMULACJI (DLA ZADANIA VDss_Lombardo - {int(percentage*100)}%)")
    print("="*50)
    print(f" METRYKA    |  MODEL STL ({int(percentage*100)}% danych)  |  MODEL MTL ({int(percentage*100)}% danych + Lipo)")
    print("-"*50)
    print(f" RMSE       |          {stl_vdss_metrics['rmse']:.4f}          |          {mtl_vdss_metrics['rmse']:.4f}")
    print(f" MAE        |          {stl_vdss_metrics['mae']:.4f}          |          {mtl_vdss_metrics['mae']:.4f}")
    print(f" R2         |          {stl_vdss_metrics['r2']:.4f}          |          {mtl_vdss_metrics['r2']:.4f}")
    print("="*50)

    with open(filepath_sim, 'a') as f:
        f.write(f"\n\n{'='*50}\nEKSPERYMENT KONTROLNY: SYMULACJA GŁODU DANYCH ({int(percentage*100)}% VDss_Lombardo)\n{'='*50}\n")
        f.write(f"  [STL VDss_Lombardo {int(percentage*100)}%] RMSE: {stl_vdss_metrics['rmse']:.4f} | MAE: {stl_vdss_metrics['mae']:.4f} | R2: {stl_vdss_metrics['r2']:.4f}\n")
        f.write(f"  [MTL VDss_Lombardo {int(percentage*100)}%] RMSE: {mtl_vdss_metrics['rmse']:.4f} | MAE: {mtl_vdss_metrics['mae']:.4f} | R2: {mtl_vdss_metrics['r2']:.4f}\n")
        f.write(f"{'='*50}\n")

print(f"\nWszystkie wyniki symulacji dla VDss_Lombardo zostały pomyślnie dopisane do pliku:\n{filepath_sim}")


  START EMULACJI GŁODU DANYCH ( 10% VDss_Lombardo vs 100% Lipofilowość)
 -> Oryginalny rozmiar treningowy VDss_Lombardo: 904 cząsteczek
 -> Symulowany (10%) rozmiar treningowy VDss_Lombardo: 90 cząsteczek
 -> Rozmiar treningowy Lipofilowości (100%): 3360 cząsteczek
  [STL] Start treningu regresora na obciętych danych...

--- START UCZENIA (Alternating Mini-Batches MTL - Bezpieczny Strumień) ---
  Epoka   0 | Przykładowy Total Loss z kroku: 0.2554
  Epoka  20 | Przykładowy Total Loss z kroku: -0.2939
  Epoka  40 | Przykładowy Total Loss z kroku: -0.6353
  Epoka  60 | Przykładowy Total Loss z kroku: -0.9600
  Epoka  80 | Przykładowy Total Loss z kroku: -1.2515
  Epoka 100 | Przykładowy Total Loss z kroku: -1.5824
  Epoka 120 | Przykładowy Total Loss z kroku: -1.9033
  Epoka 140 | Przykładowy Total Loss z kroku: -2.1928
  Epoka 160 | Przykładowy Total Loss z kroku: -2.4767
  Epoka 180 | Przykładowy Total Loss z kroku: -2.7713
  Epoka 200 | Przykładowy Total Loss z kroku: -3.0181
  Epoka 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.utils import resample

filepath_sim = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_embeddings_sym.txt"

reg_tasks = ['Lipophilicity_AZ']
class_tasks = ['HIA_Hou']

# 1. Ładowanie oryginalnych splitów
train_hia_orig, test_hia_orig = load_split_pickle('HIA_Hou')
train_lipo_orig, test_lipo_orig = load_split_pickle('Lipophilicity_AstraZeneca')

low_data_percentages = [0.10, 0.50, 1.00]

for percentage in low_data_percentages:
    print("\n" + "="*60)
    print(f"  START EMULACJI GŁODU DANYCH ( {int(percentage*100)}% HIA_Hou vs 100% Lipofilowość)")
    print("="*60)

    # Sztuczne obcięcie zbioru TRENINGOWEGO HIA_Hou do `percentage`
    n_samples_percentage = int(len(train_hia_orig) * percentage)
    train_hia_simulated = resample(train_hia_orig, replace=False, n_samples=n_samples_percentage, random_state=42)

    print(f" -> Oryginalny rozmiar treningowy HIA_Hou: {len(train_hia_orig)} cząsteczek")
    print(f" -> Symulowany ({int(percentage*100)}%) rozmiar treningowy HIA_Hou: {len(train_hia_simulated)} cząsteczek")
    print(f" -> Rozmiar treningowy Lipofilowości (100%): {len(train_lipo_orig)} cząsteczek")

    # 2. Przygotowanie czystych słowników pod model MTL
    X_train_dict, y_train_dict = prepare_clean_dicts_for_alternating(
        [train_lipo_orig, train_hia_simulated],
        reg_tasks + class_tasks,
        ['Lipophilicity_AstraZeneca', 'HIA_Hou']
    )
    # Testowe zbiory danych pozostają takie same dla wszystkich eksperymentów
    X_test_dict,  y_test_dict  = prepare_clean_dicts_for_alternating(
        [test_lipo_orig, test_hia_orig],
        reg_tasks + class_tasks,
        ['Lipophilicity_AstraZeneca', 'HIA_Hou']
    )

    # =====================================================================
    # KROK A: Uruchomienie modelu STL (Single-Task) na małych danych
    # =====================================================================
    X_tr_hia_stl = X_train_dict['HIA_Hou']
    y_tr_hia_stl = y_train_dict['HIA_Hou']
    X_te_hia_stl = X_test_dict['HIA_Hou']
    y_te_hia_stl = y_test_dict['HIA_Hou']

    stl_hia_metrics = train_STL_classifier_low_data(X_tr_hia_stl, y_tr_hia_stl, X_te_hia_stl, y_te_hia_stl, epochs=300)


    # =====================================================================
    # KROK B: Uruchomienie Twojego modelu MTL (Alternating Batches)
    # =====================================================================
    mtl_results = train_MTL_alternating_batches(X_train_dict, y_train_dict, X_test_dict, y_test_dict, reg_tasks, class_tasks, batch_size=64, epochs=300)


    # =====================================================================
    # KROK C: BEZPOŚREDNIE PORÓWNANIE I RAPORT
    # =====================================================================
    mtl_hia_metrics = mtl_results['class_tasks']['HIA_Hou']

    print("\n" + "="*50)
    print(f"   WYNIKI KOŃCOWE SYMULACJI (DLA ZADANIA HIA_Hou - {int(percentage*100)}%)")
    print("="*50)
    print(f" METRYKA    |  MODEL STL ({int(percentage*100)}% danych)  |  MODEL MTL ({int(percentage*100)}% danych + Lipo)")
    print("-"*50)
    print(f" AUROC      |          {stl_hia_metrics['auroc']:.4f}          |          {mtl_hia_metrics['auroc']:.4f}")
    print(f" F1-Score   |          {stl_hia_metrics['f1']:.4f}          |          {mtl_hia_metrics['f1']:.4f}")
    print(f" Accuracy   |          {stl_hia_metrics['accuracy']:.4f}          |          {mtl_hia_metrics['accuracy']:.4f}")
    print("="*50)

    with open(filepath_sim, 'a') as f:
        f.write(f"\n\n{'='*50}\nEKSPERYMENT KONTROLNY: SYMULACJA GŁODU DANYCH ({int(percentage*100)}% HIA_Hou)\n{'='*50}\n")
        f.write(f"  [STL HIA_Hou {int(percentage*100)}%] AUROC: {stl_hia_metrics['auroc']:.4f} | F1: {stl_hia_metrics['f1']:.4f} | ACC: {stl_hia_metrics['accuracy']:.4f}\n")
        f.write(f"  [MTL HIA_Hou {int(percentage*100)}%] AUROC: {mtl_hia_metrics['auroc']:.4f} | F1: {mtl_hia_metrics['f1']:.4f} | ACC: {mtl_hia_metrics['accuracy']:.4f}\n")
        f.write(f"{'='*50}\n")

print(f"\nWszystkie wyniki symulacji dla HIA_Hou zostały pomyślnie dopisane do pliku:\n{filepath_sim}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.utils import resample

filepath_sim = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_embeddings_sym.txt"

reg_tasks = ['Half_Life_Obach', 'Lipophilicity_AZ']
class_tasks = []

# 1. Ładowanie oryginalnych splitów
train_halflife_orig, test_halflife_orig = load_split_pickle('Half_Life_Obach')
train_lipo_orig, test_lipo_orig = load_split_pickle('Lipophilicity_AstraZeneca')

low_data_percentages = [0.10, 0.50, 1.00]

for percentage in low_data_percentages:
    print("\n" + "="*60)
    print(f"  START EMULACJI GŁODU DANYCH ( {int(percentage*100)}% Half_Life_Obach vs 100% Lipofilowość)")
    print("="*60)

    # Sztuczne obcięcie zbioru TRENINGOWEGO Half_Life_Obach do `percentage`
    n_samples_percentage = int(len(train_halflife_orig) * percentage)
    train_halflife_simulated = resample(train_halflife_orig, replace=False, n_samples=n_samples_percentage, random_state=42)

    print(f" -> Oryginalny rozmiar treningowy Half_Life_Obach: {len(train_halflife_orig)} cząsteczek")
    print(f" -> Symulowany ({int(percentage*100)}%) rozmiar treningowy Half_Life_Obach: {len(train_halflife_simulated)} cząsteczek")
    print(f" -> Rozmiar treningowy Lipofilowości (100%): {len(train_lipo_orig)} cząsteczek")

    # 2. Przygotowanie czystych słowników pod model MTL
    X_train_dict, y_train_dict = prepare_clean_dicts_for_alternating(
        [train_halflife_simulated, train_lipo_orig],
        reg_tasks + class_tasks,
        ['Half_Life_Obach', 'Lipophilicity_AstraZeneca']
    )
    # Testowe zbiory danych pozostają takie same dla wszystkich eksperymentów
    X_test_dict,  y_test_dict  = prepare_clean_dicts_for_alternating(
        [test_halflife_orig, test_lipo_orig],
        reg_tasks + class_tasks,
        ['Half_Life_Obach', 'Lipophilicity_AstraZeneca']
    )

    # =====================================================================
    # KROK A: Uruchomienie modelu STL (Single-Task) na małych danych
    # =====================================================================
    X_tr_halflife_stl = X_train_dict['Half_Life_Obach']
    y_tr_halflife_stl = y_train_dict['Half_Life_Obach']
    X_te_halflife_stl = X_test_dict['Half_Life_Obach']
    y_te_halflife_stl = y_test_dict['Half_Life_Obach']

    stl_halflife_metrics = train_STL_regressor_low_data(X_tr_halflife_stl, y_tr_halflife_stl, X_te_halflife_stl, y_te_halflife_stl, epochs=300, batch_size=64)


    # =====================================================================
    # KROK B: Uruchomienie Twojego modelu MTL (Alternating Batches)
    # =====================================================================
    mtl_results = train_MTL_alternating_batches(X_train_dict, y_train_dict, X_test_dict, y_test_dict, reg_tasks, class_tasks, batch_size=64, epochs=300)


    # =====================================================================
    # KROK C: BEZPOŚREDNIE PORÓWNANIE I RAPORT
    # =====================================================================
    mtl_halflife_metrics = mtl_results['reg_tasks']['Half_Life_Obach']

    print("\n" + "="*50)
    print(f"   WYNIKI KOŃCOWE SYMULACJI (DLA ZADANIA Half_Life_Obach - {int(percentage*100)}%)")
    print("="*50)
    print(f" METRYKA    |  MODEL STL ({int(percentage*100)}% danych)  |  MODEL MTL ({int(percentage*100)}% danych + Lipo)")
    print("-"*50)
    print(f" RMSE       |          {stl_halflife_metrics['rmse']:.4f}          |          {mtl_halflife_metrics['rmse']:.4f}")
    print(f" MAE        |          {stl_halflife_metrics['mae']:.4f}          |          {mtl_halflife_metrics['mae']:.4f}")
    print(f" R2         |          {stl_halflife_metrics['r2']:.4f}          |          {mtl_halflife_metrics['r2']:.4f}")
    print("="*50)

    with open(filepath_sim, 'a') as f:
        f.write(f"\n\n{'='*50}\nEKSPERYMENT KONTROLNY: SYMULACJA GŁODU DANYCH ({int(percentage*100)}% Half_Life_Obach)\n{'='*50}\n")
        f.write(f"  [STL Half_Life_Obach {int(percentage*100)}%] RMSE: {stl_halflife_metrics['rmse']:.4f} | MAE: {stl_halflife_metrics['mae']:.4f} | R2: {stl_halflife_metrics['r2']:.4f}\n")
        f.write(f"  [MTL Half_Life_Obach {int(percentage*100)}%] RMSE: {mtl_halflife_metrics['rmse']:.4f} | MAE: {mtl_halflife_metrics['mae']:.4f} | R2: {mtl_halflife_metrics['r2']:.4f}\n")
        f.write(f"{'='*50}\n")

print(f"\nWszystkie wyniki symulacji dla Half_Life_Obach zostały pomyślnie dopisane do pliku:\n{filepath_sim}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.utils import resample

filepath_sim = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_embeddings_sym.txt"

reg_tasks = ['Clearance_Hepatocyte_AZ', 'Lipophilicity_AZ']
class_tasks = []

# 1. Ładowanie oryginalnych splitów
train_clearance_orig, test_clearance_orig = load_split_pickle('Clearance_Hepatocyte_AZ')
train_lipo_orig, test_lipo_orig = load_split_pickle('Lipophilicity_AstraZeneca')

low_data_percentages = [0.10, 0.50, 1.00]

for percentage in low_data_percentages:
    print("\n" + "="*60)
    print(f"  START EMULACJI GŁODU DANYCH ( {int(percentage*100)}% Clearance_Hepatocyte_AZ vs 100% Lipofilowość)")
    print("="*60)

    # Sztuczne obcięcie zbioru TRENINGOWEGO Clearance_Hepatocyte_AZ do `percentage`
    n_samples_percentage = int(len(train_clearance_orig) * percentage)
    train_clearance_simulated = resample(train_clearance_orig, replace=False, n_samples=n_samples_percentage, random_state=42)

    print(f" -> Oryginalny rozmiar treningowy Clearance_Hepatocyte_AZ: {len(train_clearance_orig)} cząsteczek")
    print(f" -> Symulowany ({int(percentage*100)}%) rozmiar treningowy Clearance_Hepatocyte_AZ: {len(train_clearance_simulated)} cząsteczek")
    print(f" -> Rozmiar treningowy Lipofilowości (100%): {len(train_lipo_orig)} cząsteczek")

    # 2. Przygotowanie czystych słowników pod model MTL
    X_train_dict, y_train_dict = prepare_clean_dicts_for_alternating(
        [train_clearance_simulated, train_lipo_orig],
        reg_tasks + class_tasks,
        ['Clearance_Hepatocyte_AZ', 'Lipophilicity_AstraZeneca']
    )
    # Testowe zbiory danych pozostają takie same dla wszystkich eksperymentów
    X_test_dict,  y_test_dict  = prepare_clean_dicts_for_alternating(
        [test_lipo_orig, test_clearance_orig],
        reg_tasks + class_tasks,
        ['Lipophilicity_AstraZeneca', 'Clearance_Hepatocyte_AZ']
    )

    # =====================================================================
    # KROK A: Uruchomienie modelu STL (Single-Task) na małych danych
    # =====================================================================
    X_tr_clearance_stl = X_train_dict['Clearance_Hepatocyte_AZ']
    y_tr_clearance_stl = y_train_dict['Clearance_Hepatocyte_AZ']
    X_te_clearance_stl = X_test_dict['Clearance_Hepatocyte_AZ']
    y_te_clearance_stl = y_test_dict['Clearance_Hepatocyte_AZ']

    stl_clearance_metrics = train_STL_regressor_low_data(X_tr_clearance_stl, y_tr_clearance_stl, X_te_clearance_stl, y_te_clearance_stl, epochs=300, batch_size=64)


    # =====================================================================
    # KROK B: Uruchomienie Twojego modelu MTL (Alternating Batches)
    # =====================================================================
    mtl_results = train_MTL_alternating_batches(X_train_dict, y_train_dict, X_test_dict, y_test_dict, reg_tasks, class_tasks, batch_size=64, epochs=300)


    # =====================================================================
    # KROK C: BEZPOŚREDNIE PORÓWNANIE I RAPORT
    # =====================================================================
    mtl_clearance_metrics = mtl_results['reg_tasks']['Clearance_Hepatocyte_AZ']

    print("\n" + "="*50)
    print(f"   WYNIKI KOŃCOWE SYMULACJI (DLA ZADANIA Clearance_Hepatocyte_AZ - {int(percentage*100)}%)")
    print("="*50)
    print(f" METRYKA    |  MODEL STL ({int(percentage*100)}% danych)  |  MODEL MTL ({int(percentage*100)}% danych + Lipo)")
    print("-"*50)
    print(f" RMSE       |          {stl_clearance_metrics['rmse']:.4f}          |          {mtl_clearance_metrics['rmse']:.4f}")
    print(f" MAE        |          {stl_clearance_metrics['mae']:.4f}          |          {mtl_clearance_metrics['mae']:.4f}")
    print(f" R2         |          {stl_clearance_metrics['r2']:.4f}          |          {mtl_clearance_metrics['r2']:.4f}")
    print("="*50)

    with open(filepath_sim, 'a') as f:
        f.write(f"\n\n{'='*50}\nEKSPERYMENT KONTROLNY: SYMULACJA GŁODU DANYCH ({int(percentage*100)}% Clearance_Hepatocyte_AZ)\n{'='*50}\n")
        f.write(f"  [STL Clearance_Hepatocyte_AZ {int(percentage*100)}%] RMSE: {stl_clearance_metrics['rmse']:.4f} | MAE: {stl_clearance_metrics['mae']:.4f} | R2: {stl_clearance_metrics['r2']:.4f}\n")
        f.write(f"  [MTL Clearance_Hepatocyte_AZ {int(percentage*100)}%] RMSE: {mtl_clearance_metrics['rmse']:.4f} | MAE: {mtl_clearance_metrics['mae']:.4f} | R2: {mtl_clearance_metrics['r2']:.4f}\n")
        f.write(f"{'='*50}\n")

print(f"\nWszystkie wyniki symulacji dla Clearance_Hepatocyte_AZ zostały pomyślnie dopisane do pliku:\n{filepath_sim}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.utils import resample

filepath_sim = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_embeddings_sym.txt"

reg_tasks = ['Lipophilicity_AZ']
class_tasks = ['CYP3A4_Veith']

# 1. Ładowanie oryginalnych splitów
train_cyp3a4_orig, test_cyp3a4_orig = load_split_pickle('CYP3A4_Veith')
train_lipo_orig, test_lipo_orig = load_split_pickle('Lipophilicity_AstraZeneca')

low_data_percentages = [0.10, 0.50, 1.00]

for percentage in low_data_percentages:
    print("\n" + "="*60)
    print(f"  START EMULACJI GŁODU DANYCH ( {int(percentage*100)}% CYP3A4_Veith vs 100% Lipofilowość)")
    print("="*60)

    # Sztuczne obcięcie zbioru TRENINGOWEGO CYP3A4_Veith do `percentage`
    n_samples_percentage = int(len(train_cyp3a4_orig) * percentage)
    train_cyp3a4_simulated = resample(train_cyp3a4_orig, replace=False, n_samples=n_samples_percentage, random_state=42)

    print(f" -> Oryginalny rozmiar treningowy CYP3A4_Veith: {len(train_cyp3a4_orig)} cząsteczek")
    print(f" -> Symulowany ({int(percentage*100)}%) rozmiar treningowy CYP3A4_Veith: {len(train_cyp3a4_simulated)} cząsteczek")
    print(f" -> Rozmiar treningowy Lipofilowości (100%): {len(train_lipo_orig)} cząsteczek")

    # 2. Przygotowanie czystych słowników pod model MTL
    X_train_dict, y_train_dict = prepare_clean_dicts_for_alternating(
        [train_lipo_orig, train_cyp3a4_simulated],
        reg_tasks + class_tasks,
        ['Lipophilicity_AstraZeneca', 'CYP3A4_Veith']
    )
    # Testowe zbiory danych pozostają takie same dla wszystkich eksperymentów
    X_test_dict,  y_test_dict  = prepare_clean_dicts_for_alternating(
        [test_lipo_orig, test_cyp3a4_orig],
        reg_tasks + class_tasks,
        ['Lipophilicity_AstraZeneca', 'CYP3A4_Veith']
    )

    # =====================================================================
    # KROK A: Uruchomienie modelu STL (Single-Task) na małych danych
    # =====================================================================
    X_tr_cyp3a4_stl = X_train_dict['CYP3A4_Veith']
    y_tr_cyp3a4_stl = y_train_dict['CYP3A4_Veith']
    X_te_cyp3a4_stl = X_test_dict['CYP3A4_Veith']
    y_te_cyp3a4_stl = y_test_dict['CYP3A4_Veith']

    stl_cyp3a4_metrics = train_STL_classifier_low_data(X_tr_cyp3a4_stl, y_tr_cyp3a4_stl, X_te_cyp3a4_stl, y_te_cyp3a4_stl, epochs=300)


    # =====================================================================
    # KROK B: Uruchomienie Twojego modelu MTL (Alternating Batches)
    # =====================================================================
    mtl_results = train_MTL_alternating_batches(X_train_dict, y_train_dict, X_test_dict, y_test_dict, reg_tasks, class_tasks, batch_size=64, epochs=300)


    # =====================================================================
    # KROK C: BEZPOŚREDNIE PORÓWNANIE I RAPORT
    # =====================================================================
    mtl_cyp3a4_metrics = mtl_results['class_tasks']['CYP3A4_Veith']

    print("\n" + "="*50)
    print(f"   WYNIKI KOŃCOWE SYMULACJI (DLA ZADANIA CYP3A4_Veith - {int(percentage*100)}%)")
    print("="*50)
    print(f" METRYKA    |  MODEL STL ({int(percentage*100)}% danych)  |  MODEL MTL ({int(percentage*100)}% danych + Lipo)")
    print("-"*50)
    print(f" AUROC      |          {stl_cyp3a4_metrics['auroc']:.4f}          |          {mtl_cyp3a4_metrics['auroc']:.4f}")
    print(f" F1-Score   |          {stl_cyp3a4_metrics['f1']:.4f}          |          {mtl_cyp3a4_metrics['f1']:.4f}")
    print(f" Accuracy   |          {stl_cyp3a4_metrics['accuracy']:.4f}          |          {mtl_cyp3a4_metrics['accuracy']:.4f}")
    print("="*50)

    with open(filepath_sim, 'a') as f:
        f.write(f"\n\n{'='*50}\nEKSPERYMENT KONTROLNY: SYMULACJA GŁODU DANYCH ({int(percentage*100)}% CYP3A4_Veith)\n{'='*50}\n")
        f.write(f"  [STL CYP3A4_Veith {int(percentage*100)}%] AUROC: {stl_cyp3a4_metrics['auroc']:.4f} | F1: {stl_cyp3a4_metrics['f1']:.4f} | ACC: {stl_cyp3a4_metrics['accuracy']:.4f}\n")
        f.write(f"  [MTL CYP3A4_Veith {int(percentage*100)}%] AUROC: {mtl_cyp3a4_metrics['auroc']:.4f} | F1: {mtl_cyp3a4_metrics['f1']:.4f} | ACC: {mtl_cyp3a4_metrics['accuracy']:.4f}\n")
        f.write(f"{'='*50}\n")

print(f"\nWszystkie wyniki symulacji dla CYP3A4_Veith zostały pomyślnie dopisane do pliku:\n{filepath_sim}")

In [20]:
import numpy as np
import pandas as pd
from sklearn.utils import resample

filepath_sim = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_embeddings_sym.txt"

reg_tasks = ['Solubility_AqSolDB', 'Lipophilicity_AZ']
class_tasks = []

# 1. Ładowanie oryginalnych splitów
train_solubility_orig, test_solubility_orig = load_split_pickle('Solubility_AqSolDB')
train_lipo_orig, test_lipo_orig = load_split_pickle('Lipophilicity_AstraZeneca')

low_data_percentages = [0.10, 0.50, 1.00]

for percentage in low_data_percentages:
    print("\n" + "="*60)
    print(f"  START EMULACJI GŁODU DANYCH ( {int(percentage*100)}% Solubility_AqSolDB vs 100% Lipofilowość)")
    print("="*60)

    # Sztuczne obcięcie zbioru TRENINGOWEGO Solubility_AqSolDB do `percentage`
    n_samples_percentage = int(len(train_solubility_orig) * percentage)
    train_solubility_simulated = resample(train_solubility_orig, replace=False, n_samples=n_samples_percentage, random_state=42)

    print(f" -> Oryginalny rozmiar treningowy Solubility_AqSolDB: {len(train_solubility_orig)} cząsteczek")
    print(f" -> Symulowany ({int(percentage*100)}%) rozmiar treningowy Solubility_AqSolDB: {len(train_solubility_simulated)} cząsteczek")
    print(f" -> Rozmiar treningowy Lipofilowości (100%): {len(train_lipo_orig)} cząsteczek")

    # 2. Przygotowanie czystych słowników pod model MTL
    X_train_dict, y_train_dict = prepare_clean_dicts_for_alternating(
        [train_solubility_simulated, train_lipo_orig],
        reg_tasks + class_tasks,
        ['Solubility_AqSolDB', 'Lipophilicity_AstraZeneca']
    )
    # Testowe zbiory danych pozostają takie same dla wszystkich eksperymentów
    X_test_dict,  y_test_dict  = prepare_clean_dicts_for_alternating(
        [test_solubility_orig, test_lipo_orig],
        reg_tasks + class_tasks,
        ['Solubility_AqSolDB', 'Lipophilicity_AstraZeneca']
    )

    # =====================================================================
    # KROK A: Uruchomienie modelu STL (Single-Task) na małych danych
    # =====================================================================
    X_tr_solubility_stl = X_train_dict['Solubility_AqSolDB']
    y_tr_solubility_stl = y_train_dict['Solubility_AqSolDB']
    X_te_solubility_stl = X_test_dict['Solubility_AqSolDB']
    y_te_solubility_stl = y_test_dict['Solubility_AqSolDB']

    stl_solubility_metrics = train_STL_regressor_low_data(X_tr_solubility_stl, y_tr_solubility_stl, X_te_solubility_stl, y_te_solubility_stl)


    # =====================================================================
    # KROK B: Uruchomienie Twojego modelu MTL (Alternating Batches)
    # =====================================================================
    mtl_results = train_MTL_alternating_batches(X_train_dict, y_train_dict, X_test_dict, y_test_dict, reg_tasks, class_tasks, batch_size=64, epochs=300)


    # =====================================================================
    # KROK C: BEZPOŚREDNIE PORÓWNANIE I RAPORT
    # =====================================================================
    mtl_solubility_metrics = mtl_results['reg_tasks']['Solubility_AqSolDB']

    print("\n" + "="*50)
    print(f"   WYNIKI KOŃCOWE SYMULACJI (DLA ZADANIA Solubility_AqSolDB - {int(percentage*100)}%)")
    print("="*50)
    print(f" METRYKA    |  MODEL STL ({int(percentage*100)}% danych)  |  MODEL MTL ({int(percentage*100)}% danych + Lipo)")
    print("-"*50)
    print(f" RMSE       |          {stl_solubility_metrics['test_metrics']['rmse']:.4f}          |          {mtl_solubility_metrics['rmse']:.4f}")
    print(f" MAE        |          {stl_solubility_metrics['test_metrics']['mae']:.4f}          |          {mtl_solubility_metrics['mae']:.4f}")
    print(f" R2         |          {stl_solubility_metrics['test_metrics']['r2']:.4f}          |          {mtl_solubility_metrics['r2']:.4f}")
    print("="*50)

    with open(filepath_sim, 'a') as f:
        f.write(f"\n\n{'='*50}\nEKSPERYMENT KONTROLNY: SYMULACJA GŁODU DANYCH ({int(percentage*100)}% Solubility_AqSolDB)\n{'='*50}\n")
        f.write(f"  [STL Solubility_AqSolDB {int(percentage*100)}%] RMSE: {stl_solubility_metrics['test_metrics']['rmse']:.4f} | MAE: {stl_solubility_metrics['test_metrics']['mae']:.4f} | R2: {stl_solubility_metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"  [MTL Solubility_AqSolDB {int(percentage*100)}%] RMSE: {mtl_solubility_metrics['rmse']:.4f} | MAE: {mtl_solubility_metrics['mae']:.4f} | R2: {mtl_solubility_metrics['r2']:.4f}\n")
        f.write(f"{'='*50}\n")

print(f"\nWszystkie wyniki symulacji dla Solubility_AqSolDB zostały pomyślnie dopisane do pliku:\n{filepath_sim}")


  START EMULACJI GŁODU DANYCH ( 10% Solubility_AqSolDB vs 100% Lipofilowość)
 -> Oryginalny rozmiar treningowy Solubility_AqSolDB: 7986 cząsteczek
 -> Symulowany (10%) rozmiar treningowy Solubility_AqSolDB: 798 cząsteczek
 -> Rozmiar treningowy Lipofilowości (100%): 3360 cząsteczek
[INFO] Normalizacja danych wyjściowych...

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.5795
  Epoka  20 | Błąd MSE: 0.3433
  Epoka  40 | Błąd MSE: 0.2187
  Epoka  60 | Błąd MSE: 0.1379
  Epoka  80 | Błąd MSE: 0.0738

--- EWALUACJA ---

--- START UCZENIA (Alternating Mini-Batches MTL - Bezpieczny Strumień) ---
  Epoka   0 | Przykładowy Total Loss z kroku: 0.3468
  Epoka  20 | Przykładowy Total Loss z kroku: -0.2977
  Epoka  40 | Przykładowy Total Loss z kroku: -0.6432
  Epoka  60 | Przykładowy Total Loss z kroku: -0.9634
  Epoka  80 | Przykładowy Total Loss z kroku: -1.2702
  Epoka 100 | Przykładowy Total Loss z kroku: -1.5745
  Epoka 120 | Przykładowy Total Loss z kroku: -1.8770
  Epoka 140 | Przykłado